# NUMERICAL FEATURE GENERATOR (NFG)

In this workflow, we will process airborne SAR data that allows us to calculate surface roughness at high resolutions, as well as to identiffy various volcanic deposits.

In [ ]:
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import os
import rasterio as rio
import sys
import glob
sys.path.insert(0,"/home/jovyan/shared/Libraries")
import victor

### First, please specify the ASAR file you would like to convert from amplitude to backscatter. 

Additionally, provide the corresponding text file that contains the incidence angles

In [1]:
input_file = "ASAR_L/ASAR_L_JOINT_FP_ID27124_LINE04_RUN01_161219_LEVEL2_HH_V1.3.tif"

angle_file = "ASAR_L/ASAR_L_S_JOINT_FP_ID27124_LINE04_RUN01_161219_LEVEL2_GRID_V1.3.txt"

In [ ]:
f = open("nfg.py","r+")
conf = f.readlines()
conf[87] = f'INPUT_PATH              = "{input_file}"\n'
conf[88] = f'ANGLE_FILE              = "{angle_file}"\n'
conf[89] = 'PINCH_INPUT             = False\n'
conf[90] = "AVERAGE_INPUTS          = False\n"
conf[91] = "CONVERT_INPUT_TO_BS     = True\n"
conf[92] = "USE_OG_BS_EQN           = False\n"
conf[93] = "IMAGE_NOISE_BIAS        = 1.76243\n"
conf[94] = "SPLIT_OUTPUT            = False\n"
conf[95] = "SPLIT_THRESHOLD         = 0\n"
conf[97] = "COHERENT_OUTPUT         = False\n"
conf[98] = "RESCALE_OUTPUT          = False\n"
conf[99] = "CLUSTER_OUTPUT          = False\n"
conf[100] = "CLASS_BOUNDARY          = 0\n"
conf[102] = "DUMP_DATA               = False\n"
conf[103] = "DUMP_NODATA             = False\n"
conf[106] = "ROUGHNESS_MODEL         = None\n"
conf[107] = "WINDOWS                 = [3]\n"
conf[108] = "APPLY_GAUSSIAN_BLUR     = False\n"
conf[109] = "BLUR_FACTOR             = 1/np.sqrt(2*np.pi)\n"
conf[110] = "REPORT_AS_LOG10         = False\n"
conf[111] = "POPULATION_STATISTICS   = True\n"
conf[114] = "INTENSITY_WEIGHTED      = True\n"
conf[115] = "PINCH_WEIGHTS           = True\n"
conf[116] = "PINCH_FACTOR            = 2.0\n"
conf[119] = "NODATA_VALUE            = -34028\n"
conf[120] = "NODATA_ANGLE            = -9999.0\n"
conf[121] = "AVOID_BLANKS            = True\n"
conf[122] = "WORKAROUND_BLANKS       = False\n"
conf[125] = "MULTIPROCESSING         = True   \n"
f.seek(0)
f.writelines(conf)
f.truncate()
f.close()

In [ ]:
!python nfg.py

#### Because the Image Noise Bias values were applied, as well as Incidence Angle correction, some pixels may turn blank. To correct this, we run MODEL 9 at Window [3] to fill-in the blanks by ‘smoothing’ the image using the nearest neighbors method.

From here on, we recommend using only a portion of the ASAR strip to an area of interest, as it will significantly reduce processing time.

In [ ]:
input_file_bs = "ASAR_L/ASAR_L_JOINT_FP_ID27124_LINE04_RUN01_161219_LEVEL2_HH_V1.3_backscatter_IWp.tif"

In [ ]:
f = open("nfg.py","r+")
conf = f.readlines()
conf[87] = f'INPUT_PATH              = "{input_file_bs}"\n'
conf[88] = f'ANGLE_FILE              = "None"\n'
conf[91] = "CONVERT_INPUT_TO_BS     = False\n"
conf[92] = "USE_OG_BS_EQN           = False\n"
conf[93] = "IMAGE_NOISE_BIAS        = 0\n""
conf[102] = "DUMP_DATA               = False\n"
conf[106] = "ROUGHNESS_MODEL         = 9\n"
conf[114] = "INTENSITY_WEIGHTED      = False\n"
conf[115] = "PINCH_WEIGHTS           = False\n"
conf[116] = "PINCH_FACTOR            = 2.0\n"
f.seek(0)
f.writelines(conf)
f.truncate()
f.close()

### Our next step is to convert the backscatter data into roughness.

In [ ]:
f = open("nfg.py","r+")
conf = f.readlines()
conf[87] = f'INPUT_PATH              = "{input_file}"\n'
conf[98] = "RESCALE_OUTPUT          = True\n"
conf[102] = "DUMP_DATA               = False\n"
conf[106] = "ROUGHNESS_MODEL         = 7\n"
conf[107] = "WINDOWS                 = [15]\n"
conf[110] = "REPORT_AS_LOG10         = False\n"
conf[111] = "POPULATION_STATISTICS   = True\n"
conf[114] = "INTENSITY_WEIGHTED      = False\n"
f.seek(0)
f.writelines(conf)
f.truncate()
f.close()

### Finally, we can take the rougness data and split the output based on a threshold.

In [ ]:
input_file_rough = 

threshold = .5

In [ ]:
f = open("nfg.py","r+")
conf = f.readlines()
conf[87] = f'INPUT_PATH              = "{input_file_rough}"\n'
conf[94] = "SPLIT_OUTPUT            = True\n"
conf[95] = f"SPLIT_THRESHOLD         = {threshold}\n"
conf[98] = "RESCALE_OUTPUT          = True\n"
conf[102] = "DUMP_DATA               = False\n"
conf[106] = "ROUGHNESS_MODEL         = None\n"
conf[107] = "WINDOWS                 = [15]\n"
conf[115] = "PINCH_WEIGHTS           = True\n"
conf[116] = "PINCH_FACTOR            = 2.0\n"
f.seek(0)
f.writelines(conf)
f.truncate()
f.close()

### Now that the processing is done, we can create a dual-band map.

This will highlight roughness features visible in L-band alone (e.g., debris avalanche in MSH_TEST) and wet channel within lahar deposits, visible only in S-band.

In [4]:
import rasterio as rio

In [ ]:
merged = rio.merge.merge(l_band, s_band)
plt.imshow(merged)